# 18 · ARIMA y SARIMAX con datos solares
Greta Solange Díaz Vega · Seminario de Tesis II · UNI

Comparación retrospectiva con parámetros fijos ajustados hasta diciembre de 2009. ARIMA(2,0,0) y su extensión con SSN usan la misma dinámica, sin componente estacional. El regresor solar se retrasa según el horizonte para evitar utilizar observaciones futuras. La especificación es una referencia inicial.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]

horizontes = [3,6,12,24]
numero = 18
print(df.index.min().date(), df.index.max().date(), len(df), "meses")

1950-01-01 2026-05-01 917 meses


## Análisis

In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
filas = []
for indice in ["ICEN","ONI"]:
    serie = df[indice].to_numpy()
    for h in horizontes:
        # El regresor en el mes objetivo s es SSN(s-h-1), conocido al emitir el pronóstico.
        ssn = df.SSN.shift(h+1)
        inicio = h+1
        fin = df.index.get_loc("2010-01-01")
        media, desvio = ssn.iloc[inicio:fin].mean(), ssn.iloc[inicio:fin].std()
        z = ((ssn-media)/desvio).to_numpy()
        for solar in [False,True]:
            exog = z[inicio:fin,None] if solar else None
            modelo = SARIMAX(serie[inicio:fin], exog=exog, order=(2,0,0),trend="c")
            resultado = modelo.fit(disp=False,maxiter=200)
            if not resultado.mle_retvals["converged"]:
                raise RuntimeError("El ajuste requiere revisión")
            par = dict(zip(resultado.param_names,resultado.params))
            beta = par.get("x1",0)
            residuo = serie-beta*z
            for t in range(fin,len(df)-h):
                # AR(2) sobre el índice ajustado por el regresor. Última observación en t-1.
                previo, ultimo = residuo[t-2:t]
                for paso in range(h+1):
                    siguiente = par["intercept"] + par["ar.L1"]*ultimo + par["ar.L2"]*previo
                    previo, ultimo = ultimo, siguiente
                pred = ultimo + beta*z[t+h]
                filas.append([indice,h,solar,df.index[t],serie[t+h],pred])


## Evaluación en los mismos meses
MAE es el error absoluto medio. RMSE da más peso a los errores grandes. Compara las filas con y sin SSN del mismo índice y horizonte.

In [3]:
pred = pd.DataFrame(filas,columns=["indice","h","con_SSN","origen","observado","pronostico"])
pred["destino"] = [fecha+pd.DateOffset(months=int(h)) for fecha,h in zip(pred.origen,pred.h)]
assert not pred.duplicated(["indice","h","con_SSN","origen"]).any()
assert pred.groupby(["indice","h","origen"]).size().eq(2).all()
assert np.isfinite(pred.pronostico).all()
pred["error"] = pred.pronostico-pred.observado
metricas = pred.groupby(["indice","h","con_SSN"]).error.agg(
    N="size", MAE=lambda e:e.abs().mean(), RMSE=lambda e:np.sqrt((e**2).mean()))
pred.to_csv(f"{numero}_pronosticos.csv",index=False)
metricas.to_csv(f"{numero}_metricas.csv")
print(metricas.round(3).to_string())


                     N    MAE   RMSE
indice h  con_SSN                   
ICEN   3  False    194  0.540  0.706
          True     194  0.541  0.707
       6  False    191  0.746  0.938
          True     191  0.742  0.935
       12 False    185  0.815  1.021
          True     185  0.816  1.022
       24 False    173  0.835  1.050
          True     173  0.834  1.050
ONI    3  False    194  0.375  0.464
          True     194  0.375  0.464
       6  False    191  0.556  0.707
          True     191  0.557  0.707
       12 False    185  0.628  0.808
          True     185  0.629  0.809
       24 False    173  0.606  0.796
          True     173  0.606  0.796
